# Lab 2: Extracting variables from text
Previous session we worked with tabular **structured** data. However not all data we come across in research is already well-structured into a table. Lots of medical reports are written up in a free text (or semi-structured) format. The advent of Large Language Models (LLMs) has opened a whole new range of possibilities in processing unstructured text data. In this tutorial we will work with LLMs in a programmatic context and explore the possibilities they offer.

## Exploring the dataset
First let's grab some mock data. This lab session we are working with anonymized [Radiologists Reports](https://www.kaggle.com/datasets/saadaldoaij/radiologists-reports). This dataset is as bareboned as it gets: a single-column CSV file where the only column is `TEXT`. Each row represents a complete, multi-paragraph medical report written by a radiologist.

Let's load it in:

In [2]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/SimonIlic/ML4Epidemiology/refs/heads/main/Lesson2/ReportsDATASET.csv")
df.head()

,Text
0,\nChest PA-Lat XR\n\nImaging Study\nXray Chest...
1,"EXAM(S): Chest, 2 views, frontal and lateral\n..."
2,\nExam\nXray Chest PA and Lateral\n\nDate\nXXX...
3,\nRADIOLOGY REPORT\n\nExamination\nPA and late...
4,\nChest PA-Lat XR\n\nImaging Study\nXray Chest...


Let's take a closer look at a single sample in the dataset. Adjust the number for `row` to print out different rows of the Text column. Make yourself familiar with the report style.

In [ ]:
i = 7
report = df.Text[i]
print(report)

As you can see the data has been agressively anonymised. Dates, names and sometimes even parts of diagnosis have been removed and replaced with XXXXs. Generally the reports do have some structure and consist of sections like _Indication_ _Comparison_, _Impression_ and _Findings_.

Exercise:
Browse through the dataset and figure out what the different sections aim to note. Not every report has every section.
* Findings:
* Impression:
* Comparison:
* Indication:
* Exam:

What are other common headers you have found throughout the dataset?


...

## Working with LLMs programmatically
By now you are probably familiar with AI powered chatbots like chatGPT. Google Colab allows us to interact with LLMs simply through a python function.

In [ ]:
from google.colab import ai

The `ai` module has one simple method called `generate_text`. It requires one argument, the `prompt` or message we want to send the chatbot. It then generates an answer bases on this prompt. Experiment with the method by asking anything that comes to mind.

In [ ]:
ai.generate_text("Hello, world!")

In [ ]:
ai.generate_text("What's 2 + 2?")

While this is not the best interface to interact with your daily helpful assitant, it is great for us researchers to deploy an LLM to help us process data. Interacting with a language model straight from python allows us to create structure in an unstructered dataset.

Let's try feeding the language model a radiology report and see what it thinks:

In [ ]:
print(ai.generate_text("Is this patient sick? " + report))

This is great! We have unlocked a tool to cognitively reason over the complex domain of open-text data. The problem is, the output itself is also unstructured. If we want to use this data in our research we will need to structure it like a table. Let's find a way to make our chatbot act more like a robot.

Let's start with something small. See if we can extract the sex of the patient from the reports using `ai.generate_text()`

In [ ]:
instruction = "What is the sex of the patient in this report? Answer with one of ['Male', 'Female', 'Unknown']\n"
for report in df.Text[:10]:
    prompt = instruction + report
    print(ai.generate_text(prompt))

Great we now have a way to extract features from the open-text field. Let's work this new feature back into our dataset.

In [ ]:
df['Sex'] = df.Text.apply(lambda report: ai.generate_text(instruction + report))  # Running this line will take a while

As you will notice running the above line is taking a long time. We've just learned an important lesson about AI research: It takes a lot of time to train and run inference on big machine learning models! In our previous worksheet our model had ~100 trainable parameters. Typical production level LLMs have Billions (with a B!) of parameters, resulting in hundreds of thousands of matrix multiplications when processing a single prompt.

We can use the [tqdm](https://tqdm.github.io/) module to show us a progress bar on large loops of python code. Here's an example below.

AI can make mistakes. Let's double-check the responses. If you're not happy with the results, experiment and refine your prompt. Once you're satisfied, move on.

In [ ]:
# load in the tqdm library to show a progress bar for the apply function
from tqdm import tqdm
tqdm.pandas()
# use progress_apply instead of apply to show a progress bar
df['Sex'] = df.Text.progress_apply(lambda report: ai.generate_text(instruction + report))

For teaching purposes it is not practical to wait over 1 hour, for each line to run. Go back up in the notebook and adjust the dataset size to something workable, aim for ~3 minutes processing time for each apply line.

In [ ]:
# sample 10 random rows to check the results
df.sample(10)

Now we might have gotten similar results by stringmatching "male" and "female" exactly. Let's try something a bit more challenging, where we actually have to interpret the textfield without clear signaling words. We will try to extract a specific diagnosis.

Pneumonia is a good candidate: radiologists almost never say "pneumonia" outright. The real signal is scattered across synonyms; "consolidation", "infiltrate", "airspace opacity". Additionaly, it's heavily hedged: "most concerning for pneumonia", "may represent pneumonia", "no focal opacity to suggest pneumonia". The last one is particularly tricky, pneumonia is mentioned but to negate it.

In [ ]:
instruction = """What is the pneumonia status in this chest X-ray report?
Reply with exactly one word: Confirmed, Suspected, Absent, or Unknown.

- Confirmed: radiologist explicitly states pneumonia or consolidation as the finding
- Suspected: hedged language used ("consistent with", "cannot exclude", "concerning for", "may represent")
- Absent: report explicitly rules out pneumonia, consolidation, or airspace disease
- Unknown: no relevant mention

Ignore XXXX tokens. Use the IMPRESSION section, or FINDINGS if absent.\n\n"""

In [ ]:
df['Pneumonia'] = df.Text.progress_apply(lambda report: ai.generate_text(instruction + report))

We can also try and extract a diagnosis, or radiological finding, from the data right away with a more open-ended prompt.

In [ ]:
instruction = """What is the primary radiological finding in this chest X-ray report? Answer in 5 words or fewer. Use the IMPRESSION section, or FINDINGS if absent. If normal, answer "Normal". If hedged or unclear, answer "Unknown". Ignore XXXX tokens.\n\n"""

In [ ]:
df['Diagnosis'] = df.Text.progress_apply(lambda report: ai.generate_text(instruction + report))

Have a look and see if the Pneumonia findings and the extract diagnosis concur.

In [ ]:
df[df['Pneumonia'] == 'Confirmed']

Now write 3 prompts yourself. Extract useful fields from the free text, that you might find interesting or would like to use in a research setting. Some suggestions:

* **Prior comparison available** — Was a prior imaging study available for comparison? (Look for "Comparison: None" vs an actual date or study reference)
* **Clinical indication** — Why was the imaging study ordered? (e.g. "Positive TB test", "chest pain", "preop surgery")
* **Cardiomegaly** — Does the report mention an enlarged heart? Watch out for nuance: "borderline cardiomegaly" vs "cardiac silhouette within normal limits"
* **Pleural effusion** — Is there evidence of pleural effusion?
* **Follow-up recommended** — Does the report recommend additional imaging or further workup? (e.g. "recommend followup radiograph", "recommend CT for further evaluation")

For each prompt, add the extracted column to the dataframe and spot-check a few results.

In [ ]:
# free code field for the exercise

## Working with LLMs privately
Okay, so what? Couldn't we have dragged this csv into ChatGPT and gotten it to do all this work for us? Yes, probably..
BUT in a real research scenario this would entail sending a ton of privacy sensitive medical data to a third-party. A huge data breach! Thus it is more likely that you will use LLMs locally on your own machine to process unstructured text data.

Luckily the research community has set up great infrastructure for sharing open-source models!

### 🤗 Hugging Face: The GitHub of AI Models

[Hugging Face](https://huggingface.co) is the go-to platform for sharing and downloading open-source models. Think of it like GitHub, but instead of code repositories, people share trained machine learning models. At the time of writing, it hosts hundreds of thousands of models — for text, images, audio, and more.

The core library you'll interact with is called `transformers`, and it makes loading a model as simple as a few lines of Python:
```python
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
classifier("The patient reported significant improvement after treatment.")
```

When you run this for the first time, the model weights are **downloaded and cached locally** on your machine. After that, everything runs offline. No data ever leaves your computer.

### Key concepts

- **Model hub**: The central repository at `huggingface.co/models` where you can search and filter models by task, language, size, and license.
- **`pipeline()`**: A high-level API that wraps a model and its tokenizer together, letting you go from raw text to predictions in one step.
- **Tasks**: Hugging Face organises models by task — e.g. `text-classification`, `token-classification` (NER), `summarization`, `question-answering`. Picking the right task narrows down which models are suitable for your use case.
- **Model cards**: Every model comes with a documentation page describing what it was trained on, its intended use, and known limitations. Always read these before using a model in research!

### Choosing the right model

Not all models are created equal. When browsing the hub, consider:

1. **Task fit** — was it fine-tuned for something close to your use case? A model trained on clinical notes will outperform a general-purpose one on medical text.
2. **Size** — larger models tend to be more capable, but require more RAM/VRAM and are slower. For local use on a laptop, models under ~1B parameters are a practical starting point.
3. **License** — for academic research this is often fine, but always check whether the license permits your intended use.

In [ ]:
from transformers import pipeline
generate_text = pipeline("text-generation", model="BioMistral/BioMistral-7B")

In [ ]:
generate_text("Hello, world!", max_new_tokens=50)